# Recuperar metadatos completos Urban-v1 y estimar almacenamiento (solo TRAIN/VAL)

Esta notebook recupera el artifact de GitHub Actions **14770171764**, verifica su SHA-256 y reconstruye manifiestos e índices completos de A2D2/UrbanSyn **solo para TRAIN y VAL**. Los guarda en una carpeta nueva de Drive, separada de las 40 muestras ya materializadas.

Después ejecuta el materializador exclusivamente con `--dry-run`, sin límite de muestras y sin `--require-instances`, para estimar todos los assets de los 3.751 registros.

**Seguridad:** no abre, extrae, lee, materializa ni estima TEST; no descarga imágenes/máscaras; no inicia entrenamiento, backward, optimizer.step ni checkpoints; no modifica la carpeta `urban_data_capacity` de las 40 muestras. El dry-run consulta tamaños remotos mediante metadatos/HEAD.

## 1. Montar Drive y sincronizar el repositorio privado

Añade un secreto de Colab llamado `GITHUB_TOKEN` con acceso de solo lectura a Contents y Actions para este repositorio.

In [ ]:
from google.colab import drive, userdata
from pathlib import Path
from getpass import getpass
import os, sys, json, hashlib, subprocess, shutil, zipfile, requests, datetime

drive.mount('/content/drive')
REPO = 'Leohf04/Procesamiento-de-Im-genes-'
BRANCH = 'urban-models-and-benchmark'
WORKDIR = Path('/content/Procesamiento-de-Im-genes-')

try:
    TOKEN = userdata.get('GITHUB_TOKEN')
except Exception:
    TOKEN = None
if not TOKEN:
    TOKEN = getpass('GitHub PAT (Contents: Read-only + Actions: Read-only): ')
if not TOKEN:
    raise RuntimeError('Falta GITHUB_TOKEN.')

askpass = Path('/tmp/urban_v1_git_askpass.sh')
askpass.write_text('#!/bin/sh\\ncase "$1" in\\n *Username*) echo x-access-token ;;\\n *Password*) echo "$COLAB_GITHUB_TOKEN" ;;\\n *) exit 1 ;;\\nesac\\n')
askpass.chmod(0o700)
env = os.environ.copy()
env.update({'GIT_ASKPASS': str(askpass), 'GIT_TERMINAL_PROMPT': '0', 'COLAB_GITHUB_TOKEN': TOKEN})
if WORKDIR.exists() and (WORKDIR/'.git').is_dir():
    subprocess.run(['git','-C',str(WORKDIR),'fetch','origin',BRANCH],check=True,env=env)
    subprocess.run(['git','-C',str(WORKDIR),'checkout',BRANCH],check=True)
    subprocess.run(['git','-C',str(WORKDIR),'pull','--ff-only','origin',BRANCH],check=True,env=env)
elif WORKDIR.exists():
    raise RuntimeError(f'{WORKDIR} existe pero no es un repo Git; no se sobrescribe.')
else:
    subprocess.run(['git','clone','--single-branch','--branch',BRANCH,
                    f'https://github.com/{REPO}.git',str(WORKDIR)],check=True,env=env)
del env
HEAD = subprocess.check_output(['git','-C',str(WORKDIR),'rev-parse','HEAD'],text=True).strip()
print('HEAD:', HEAD)
print('RUN_TRAINING=False')

## 2. Recuperar y verificar el artifact

Se comprueban repositorio, run, caducidad y digest publicado por GitHub. No se extrae el ZIP completo.

In [ ]:
ARTIFACT_ID = 14770171764
RUN_ID = 45038801547
EXPECTED_SHA256 = 'b6cbcb54dfd0598e6a9011667073028be1804fc1c112d82e516a42bdbf2ba4e5'
API = f'https://api.github.com/repos/{REPO}/actions/artifacts/{ARTIFACT_ID}'
headers = {'Authorization': f'Bearer {TOKEN}', 'Accept': 'application/vnd.github+json',
           'X-GitHub-Api-Version': '2022-11-28'}
meta_response = requests.get(API, headers=headers, timeout=60)
if meta_response.status_code != 200:
    raise RuntimeError(f'No se pudo leer el artifact ({meta_response.status_code}): {meta_response.text[:700]}')
meta = meta_response.json()
if meta.get('expired'):
    raise RuntimeError('El artifact ha expirado.')
if meta.get('workflow_run',{}).get('id') != RUN_ID:
    raise RuntimeError('El artifact no pertenece al run esperado.')
if meta.get('digest') != f'sha256:{EXPECTED_SHA256}':
    raise RuntimeError(f'Digest publicado inesperado: {meta.get("digest")}')
zip_response = requests.get(API+'/zip', headers=headers, timeout=300)
if zip_response.status_code != 200:
    raise RuntimeError(f'Falló descarga del ZIP ({zip_response.status_code}): {zip_response.text[:500]}')
ZIP_PATH = Path('/content/urban_annotation_artifact_14770171764.zip')
ZIP_PATH.write_bytes(zip_response.content)
actual_sha = hashlib.sha256(ZIP_PATH.read_bytes()).hexdigest()
if actual_sha != EXPECTED_SHA256:
    ZIP_PATH.unlink(missing_ok=True)
    raise RuntimeError(f'SHA-256 del ZIP no coincide: {actual_sha}')
print(json.dumps({'artifact_id':ARTIFACT_ID,'name':meta.get('name'),
                  'expired':meta.get('expired'),'size_in_bytes':len(zip_response.content),
                  'sha256':actual_sha,'expires_at':meta.get('expires_at')},indent=2))
del TOKEN, headers, meta_response, zip_response

## 3. Reconstruir metadatos completos TRAIN/VAL

Solo se abren los cuatro pares de manifiestos semánticos/instancias y los cuatro índices TRAIN/VAL. No se extrae ningún archivo TEST. Se validan filas, IDs, estadísticas y assets requeridos.

In [ ]:
import json, hashlib, zipfile
from pathlib import Path

def read_jsonl_bytes(raw):
    return [json.loads(line) for line in raw.decode('utf-8').splitlines() if line.strip()]

def write_jsonl(path, rows):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('w',encoding='utf-8') as f:
        for row in rows:
            f.write(json.dumps(row,sort_keys=True,ensure_ascii=False)+'\\n')

def member_for(zf, suffix):
    hits = [n for n in zf.namelist() if n.endswith(suffix)]
    if len(hits) != 1:
        raise RuntimeError(f'Esperaba un único miembro para {suffix}; encontrados={len(hits)}')
    return hits[0]

EXPECTED = {('a2d2','train'):1000,('a2d2','val'):1000,
            ('urbansyn','train'):1000,('urbansyn','val'):751}
ASSET_KEYS = {'a2d2':('rgb','semantic'),'urbansyn':('rgb','semantic','panoptic')}
manifest_out = {}
index_out = {}
checks = []
with zipfile.ZipFile(ZIP_PATH,'r') as zf:
    for ds in ('a2d2','urbansyn'):
        for split in ('train','val'):
            stem = f'{ds}_{split}'
            sem_name = member_for(zf,f'balanced/{ds}_semantic_{split}.jsonl')
            ins_name = member_for(zf,f'balanced/{ds}_instances_{split}.jsonl')
            ann_name = member_for(zf,f'annotation_index/{ds}_{split}.annotations.jsonl')
            sem = read_jsonl_bytes(zf.read(sem_name))
            ins = read_jsonl_bytes(zf.read(ins_name))
            ann = read_jsonl_bytes(zf.read(ann_name))
            if len(sem) != EXPECTED[(ds,split)] or len(ins) != EXPECTED[(ds,split)] or len(ann) != EXPECTED[(ds,split)]:
                raise RuntimeError(f'{stem}: conteos inesperados sem/inst/index={len(sem)}/{len(ins)}/{len(ann)}; esperado={EXPECTED[(ds,split)]}')
            def keyed(rows, label):
                ids = [str(x.get('sample_id','')) for x in rows]
                if any(not x for x in ids) or len(ids) != len(set(ids)):
                    raise RuntimeError(f'{stem}: IDs vacíos o duplicados en {label}')
                return {str(x['sample_id']):x for x in rows}
            sm, im, am = keyed(sem,'semantic'), keyed(ins,'instances'), keyed(ann,'annotation_index')
            if set(sm) != set(im) or set(sm) != set(am):
                raise RuntimeError(f'{stem}: conjuntos de IDs no coinciden entre los tres archivos')
            for sid,row in sm.items():
                if any(row.get(k) != am[sid].get(k) for k in ('semantic_pixels','instance_counts')):
                    raise RuntimeError(f'{stem}: estadísticas no coinciden con índice para {sid}')
                if any(im[sid].get(k) != row.get(k) for k in ('semantic_pixels','instance_counts')):
                    raise RuntimeError(f'{stem}: estadísticas sem/inst distintas para {sid}')
                sem_base = {k:v for k,v in row.items() if k not in ('semantic_pixels','instance_counts')}
                ins_base = {k:v for k,v in im[sid].items() if k not in ('semantic_pixels','instance_counts')}
                if sem_base != ins_base:
                    raise RuntimeError(f'{stem}: metadatos sem/inst distintos para {sid}')
                if am[sid].get('annotation_status','ok') != 'ok':
                    raise RuntimeError(f'{stem}: annotation_status no OK para {sid}')
                assets = row.get('assets',{})
                for key in ASSET_KEYS[ds]:
                    if not isinstance(assets.get(key),str) or not assets[key]:
                        raise RuntimeError(f'{stem}: asset {key} ausente para {sid}')
                if ds == 'urbansyn' and not row.get('revision'):
                    raise RuntimeError(f'{stem}: revision fijada ausente para {sid}')
            ordered_ids = sorted(sm, key=lambda sid: hashlib.sha256(f'2026:{sid}'.encode()).hexdigest())
            manifest_out[stem] = [{k:v for k,v in sm[sid].items()
                                   if k not in ('semantic_pixels','instance_counts')}
                                  for sid in ordered_ids]
            index_out[stem] = [am[sid] for sid in ordered_ids]
            checks.append({'split':stem,'manifest_rows':len(sem),'instance_rows':len(ins),
                           'index_rows':len(ann),'ids_unique':True,'ids_match':True,
                           'statistics_match':True,'annotation_status_ok':True,
                           'required_assets_present':True})
for ds in ('a2d2','urbansyn'):
    tr={str(x['sample_id']) for x in manifest_out[f'{ds}_train']}
    va={str(x['sample_id']) for x in manifest_out[f'{ds}_val']}
    overlap=tr & va
    if overlap:
        raise RuntimeError(f'{ds}: fuga TRAIN/VAL detectada ({len(overlap)} IDs)')
    checks.append({'dataset':ds,'train_val_overlap':0})

print(json.dumps(checks,indent=2,ensure_ascii=False))

## 4. Persistir manifiestos e índices completos en una carpeta nueva de Drive

No se sobrescribe `ProcesamientoImagenes/urban_data_capacity`, que contiene las 40 muestras de preflight.

In [ ]:
DRIVE_ROOT = Path('/content/drive/MyDrive/ProcesamientoImagenes')
FULL_DIR = DRIVE_ROOT/'urban_v1_full_metadata_20261124'
if FULL_DIR.exists():
    raise RuntimeError(f'{FULL_DIR} ya existe; no se sobrescribe. Cambia el sufijo de fecha/hora en FULL_DIR para crear otra copia.')
MANIFEST_DIR = FULL_DIR/'manifests'
INDEX_DIR = FULL_DIR/'annotation_index'
for stem,rows in manifest_out.items():
    write_jsonl(MANIFEST_DIR/f'{stem}.jsonl',rows)
    write_jsonl(INDEX_DIR/f'{stem}.annotations.jsonl',index_out[stem])

file_hashes = {}
for p in sorted(FULL_DIR.rglob('*.jsonl')):
    file_hashes[str(p.relative_to(FULL_DIR))] = hashlib.sha256(p.read_bytes()).hexdigest()
metadata_report = {
    'artifact_id':ARTIFACT_ID,'source_run':RUN_ID,'artifact_sha256':EXPECTED_SHA256,
    'git_head':HEAD,'created_utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),
    'expected_rows':{f'{ds}_{sp}':n for (ds,sp),n in EXPECTED.items()},
    'checks':checks,'sha256':file_hashes,'test_split_touched':False,
    'assets_downloaded':False,'training_started':False
}
(FULL_DIR/'reconstruction_report.json').write_text(json.dumps(metadata_report,indent=2,ensure_ascii=False)+'\\n')
print('Metadatos completos persistidos:',FULL_DIR)
print(json.dumps({'rows':{k:len(v) for k,v in manifest_out.items()},
                  'jsonl_files':len(file_hashes),'test_split_touched':False,
                  'assets_downloaded':False},indent=2))

## 5. Ejecutar estimación remota completa (dry-run)

El comando no usa `--max-samples-per-split` ni `--require-instances`: estima los 3.751 registros y todos sus assets requeridos. `--dry-run` evita descargar imágenes/máscaras.

In [ ]:
import importlib.util, subprocess, json, shutil
missing = [name for name in ('boto3','huggingface_hub') if importlib.util.find_spec(name) is None]
if missing:
    subprocess.run([sys.executable,'-m','pip','install','-q',*missing],check=True)

DRY_DIR = Path('/content/urban_v1_full_dry_run')
if DRY_DIR.exists():
    shutil.rmtree(DRY_DIR)
cmd = [sys.executable,'scripts/materialize_urban_experiment_data.py',
       '--manifest-dir',str(MANIFEST_DIR),
       '--annotation-index-dir',str(INDEX_DIR),
       '--output-dir',str(DRY_DIR),
       '--dry-run']
run = subprocess.run(cmd,cwd=WORKDIR,text=True,capture_output=True)
print(run.stdout[-20000:])
if run.returncode:
    print(run.stderr[-12000:])
    raise RuntimeError('Falló el dry-run completo. No se descargaron assets.')
report_path = DRY_DIR/'materialization_preflight.json'
if not report_path.is_file():
    raise RuntimeError('No se generó materialization_preflight.json.')
estimate = json.loads(report_path.read_text())
if estimate.get('status') != 'PREFLIGHT_PASS' or estimate.get('dry_run') is not True:
    raise RuntimeError('El materializador no confirmó PREFLIGHT_PASS/dry_run.')
if estimate.get('test_touched') is not False or estimate.get('training_started') is not False:
    raise RuntimeError('El reporte no confirma aislamiento de TEST y entrenamiento.')
expected_samples = sum(EXPECTED.values())
if sum(x['selected_samples'] for x in estimate['selection_validation']) != expected_samples:
    raise RuntimeError('El dry-run no cubrió los 3.751 registros esperados.')
if any(x['manifest_samples'] != EXPECTED[(x['dataset'],x['split'])] for x in estimate['selection_validation']):
    raise RuntimeError('El dry-run detectó conteos distintos del protocolo.')
shutil.copy2(report_path,FULL_DIR/'materialization_preflight_full.json')
print(json.dumps({'status':estimate['status'],
                  'samples':sum(x['selected_samples'] for x in estimate['selection_validation']),
                  'assets':estimate['asset_count'],
                  'estimated_remote_asset_bytes':estimate['estimated_remote_asset_bytes'],
                  'estimated_remote_asset_gib':estimate['estimated_remote_asset_gib'],
                  'test_touched':estimate['test_touched'],
                  'training_started':estimate['training_started'],
                  'report_in_drive':str(FULL_DIR/'materialization_preflight_full.json')},indent=2))

## 6. Bloque único de continuidad

Copia el JSON delimitado. No contiene credenciales.

In [ ]:
final_payload = {
    'status':'PASS' if estimate.get('status') == 'PREFLIGHT_PASS' else 'INCOMPLETE',
    'artifact_id':ARTIFACT_ID,'artifact_sha256':EXPECTED_SHA256,'git_head':HEAD,
    'metadata_dir':str(FULL_DIR),'metadata_sha256':file_hashes,
    'manifest_rows':{k:len(v) for k,v in manifest_out.items()},
    'reconstruction_checks':checks,
    'dry_run_status':estimate.get('status'),
    'dry_run_samples':sum(x['selected_samples'] for x in estimate['selection_validation']),
    'dry_run_asset_count':estimate.get('asset_count'),
    'estimated_remote_asset_bytes':estimate.get('estimated_remote_asset_bytes'),
    'estimated_remote_asset_gib':estimate.get('estimated_remote_asset_gib'),
    'dry_run_selection_validation':estimate.get('selection_validation'),
    'dry_run_report':str(FULL_DIR/'materialization_preflight_full.json'),
    'test_split_touched':False,'assets_downloaded':False,
    'RUN_TRAINING':False,'backward_calls':0,'optimizer_steps':0,
    'checkpoint_written':False
}
print('===== BLOQUE_CONTINUIDAD_INICIO =====')
print(json.dumps(final_payload,ensure_ascii=False,sort_keys=True))
print('===== BLOQUE_CONTINUIDAD_FIN =====')